# 01 — Thu thập dữ liệu NYC 311

Notebook này thực hiện **bước thu thập dữ liệu ban đầu** cho đề tài:

> **Phân tích đặc điểm và xu hướng yêu cầu dịch vụ công tại New York giai đoạn 2022–2025 qua dữ liệu NYC 311**

## Mục tiêu

1. Kết nối tới nguồn chính thức **NYC Open Data**.
2. Giới hạn population theo `created_date` từ **01/01/2022 đến 31/12/2025**.
3. Chỉ lấy 15 trường cần dùng cho bài phân tích.
4. Tải dữ liệu theo **tháng** để tránh giữ toàn bộ ~13.5 triệu dòng trong RAM.
5. Kiểm tra số dòng, `unique_key`, phạm vi ngày và lưu dạng **Snappy Parquet**.
6. Tải GeoJSON Community Districts phục vụ bản đồ.

> **Lưu ý:** Notebook này dùng để tạo **snapshot ban đầu 2022–2025**.  
> Việc đọc lại nguồn và cập nhật định kỳ sau đó được thực hiện bởi `dashboard/refresh_data.py`.

## 1. Cấu hình môi trường và nguồn dữ liệu

Dataset chính: **311 Service Requests from 2020 to Present**  
Dataset ID: `erm2-nwe9`

Population của bài được cố định bằng:

```text
2022-01-01 <= created_date < 2026-01-01
```

Như vậy dashboard luôn phân tích bốn năm hoàn chỉnh **2022–2025**.

In [ ]:
from pathlib import Path
import gc
import math
import os
import time

import pandas as pd
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ------------------------------------------------------------
# Project scope
# ------------------------------------------------------------

DATASET_ID = "erm2-nwe9"

BASE_URL = (
    f"https://data.cityofnewyork.us/resource/"
    f"{DATASET_ID}.json"
)

META_URL = (
    f"https://data.cityofnewyork.us/api/views/"
    f"{DATASET_ID}"
)

ANALYSIS_START = pd.Timestamp("2022-01-01 00:00:00")
ANALYSIS_END_EXCLUSIVE = pd.Timestamp("2026-01-01 00:00:00")

YEARS = [2022, 2023, 2024, 2025]

COLUMNS = [
    "unique_key",
    "created_date",
    "closed_date",
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "community_board",
    "borough",
    "status",
    "open_data_channel_type",
    "latitude",
    "longitude",
]

EXPECTED_YEAR_ROWS = {
    2022: 3_169_960,
    2023: 3_224_723,
    2024: 3_456_770,
    2025: 3_655_041,
}

EXPECTED_TOTAL_ROWS = sum(EXPECTED_YEAR_ROWS.values())


# ------------------------------------------------------------
# Output directory
# Kaggle: /kaggle/working/nyc311_raw
# Local:  <project>/data/raw
# ------------------------------------------------------------

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/nyc311_raw")
else:
    OUTPUT_DIR = Path("../data/raw").resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Expected total rows:", f"{EXPECTED_TOTAL_ROWS:,}")

## 2. Tạo HTTP session có retry

NYC Open Data có thể trả lỗi tạm thời hoặc giới hạn request.  
`Retry` giúp notebook tự thử lại với các lỗi `429`, `5xx` thay vì dừng ngay.

In [ ]:
retry_strategy = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session = requests.Session()
session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update(
    {
        "User-Agent": "NYC311-Analytics-Academic-Project/1.0",
        "Accept": "application/json",
    }
)

# Có thể đặt SOCRATA_APP_TOKEN trong environment nếu có.
app_token = os.getenv("SOCRATA_APP_TOKEN")
if app_token:
    session.headers["X-App-Token"] = app_token

print("Session ready.")

## 3. Kiểm tra kết nối tới NYC Open Data

Cell này chỉ lấy 5 bản ghi để xác nhận:

- endpoint truy cập được;
- schema trả về đúng;
- không cần tải dữ liệu lớn để kiểm tra kết nối.

In [ ]:
response = session.get(
    BASE_URL,
    params={
        "$select": ",".join(COLUMNS),
        "$limit": 5,
    },
    timeout=(10, 60),
)

print("HTTP status:", response.status_code)
response.raise_for_status()

sample = pd.DataFrame(response.json())
display(sample)

print("\nReturned columns:")
print(sample.columns.tolist())

## 4. Kiểm tra metadata của dataset

Metadata được dùng để lưu bằng chứng về **nguồn chính thức** và thời điểm dataset phía NYC được cập nhật.

In [ ]:
response = session.get(
    META_URL,
    timeout=(10, 60),
)

response.raise_for_status()
metadata = response.json()

print("Dataset name:", metadata.get("name"))
print("Dataset ID:", metadata.get("id"))
print("Rows updated at (epoch):", metadata.get("rowsUpdatedAt"))
print("Data updated at (epoch):", metadata.get("dataUpdatedAt"))
print("Metadata updated at (epoch):", metadata.get("metadataUpdatedAt"))

## 5. Hàm tạo khoảng thời gian và đếm số request theo tháng

Trước khi download, notebook hỏi API số dòng của từng tháng. Việc này có hai lợi ích:

1. biết chính xác kích thước từng partition;
2. có `expected_rows` để kiểm tra file sau khi tải.

Dữ liệu được chia thành **48 partition tháng** thay vì một file 13.5 triệu dòng.

In [ ]:
def month_bounds(year, month):
    start = pd.Timestamp(
        year=year,
        month=month,
        day=1,
    )
    end = start + pd.offsets.MonthBegin(1)
    return start, end


def to_soql_timestamp(ts):
    return pd.Timestamp(ts).strftime(
        "%Y-%m-%dT%H:%M:%S.000"
    )


def get_month_count(year, month):
    start, end = month_bounds(year, month)

    where_clause = (
        f"created_date >= '{to_soql_timestamp(start)}' "
        f"AND created_date < '{to_soql_timestamp(end)}'"
    )

    response = session.get(
        BASE_URL,
        params={
            "$select": "count(*) AS n",
            "$where": where_clause,
        },
        timeout=(10, 60),
    )

    response.raise_for_status()
    payload = response.json()

    return int(payload[0]["n"])

### Lấy bảng count cho toàn bộ 48 tháng

Cell này chỉ gọi `count(*)`, nên nhẹ hơn rất nhiều so với download dữ liệu.

In [ ]:
monthly_counts = []

for year in YEARS:
    for month in range(1, 13):
        n = get_month_count(year, month)

        monthly_counts.append(
            {
                "year": year,
                "month": month,
                "rows": n,
            }
        )

        print(
            f"{year}-{month:02d}: "
            f"{n:,}"
        )

count_table = pd.DataFrame(monthly_counts)

display(count_table)

print(
    "\nTotal source rows:",
    f"{count_table['rows'].sum():,}",
)

print(
    "Matches expected total:",
    count_table["rows"].sum()
    == EXPECTED_TOTAL_ROWS,
)

### Kiểm tra tổng số request theo năm

Các tổng tham chiếu của snapshot đã tải:

| Năm | Số request |
|---:|---:|
| 2022 | 3,169,960 |
| 2023 | 3,224,723 |
| 2024 | 3,456,770 |
| 2025 | 3,655,041 |
| **Tổng** | **13,506,494** |

Nếu nguồn được đọc lại ở thời điểm khác, một số giá trị trường của các request lịch sử có thể thay đổi; tuy nhiên population vẫn được khóa bằng `created_date`.

In [ ]:
year_count_table = (
    count_table
    .groupby("year", as_index=False)["rows"]
    .sum()
)

year_count_table["expected_rows"] = (
    year_count_table["year"]
    .map(EXPECTED_YEAR_ROWS)
)

year_count_table["matches_snapshot"] = (
    year_count_table["rows"]
    == year_count_table["expected_rows"]
)

display(year_count_table)

## 6. Hàm download một tháng

### Vì sao phân trang?

Socrata không nên được yêu cầu trả vài triệu dòng trong một response.  
Notebook dùng batch `50,000` dòng và sắp xếp:

```text
created_date ASC, unique_key ASC
```

để việc phân trang có thứ tự ổn định hơn.

### Kiểm tra sau khi tải

Mỗi partition phải thỏa:

- số dòng = count từ API;
- `unique_key` không trùng;
- `created_date` nằm đúng tháng;
- chỉ chứa 15 trường được chọn.

In [ ]:
BATCH_SIZE = 50_000


def validate_month(df, year, month, expected_rows):
    start, end = month_bounds(year, month)

    created = pd.to_datetime(
        df["created_date"],
        errors="coerce",
    )

    checks = {
        "row_count_ok": (
            len(df) == expected_rows
        ),
        "unique_key_ok": (
            df["unique_key"]
            .astype("string")
            .nunique(dropna=False)
            == len(df)
        ),
        "created_date_parse_ok": (
            created.notna().all()
        ),
        "created_date_range_ok": (
            (
                (created >= start)
                & (created < end)
            ).all()
        ),
        "schema_ok": (
            list(df.columns) == COLUMNS
        ),
    }

    if not all(checks.values()):
        raise ValueError(
            f"Validation failed "
            f"for {year}-{month:02d}: "
            f"{checks}"
        )

    return checks


def download_month(
    year,
    month,
    expected_rows=None,
    overwrite=False,
):
    if expected_rows is None:
        expected_rows = get_month_count(
            year,
            month,
        )

    output_file = (
        OUTPUT_DIR
        / f"nyc311_{year}_{month:02d}_raw.parquet"
    )

    # Nếu file đã tồn tại và không yêu cầu overwrite,
    # kiểm tra nhanh rồi dùng lại.
    if output_file.exists() and not overwrite:
        existing = pd.read_parquet(output_file)

        validate_month(
            existing,
            year,
            month,
            expected_rows,
        )

        print(
            f"SKIP {year}-{month:02d}: "
            f"{output_file.name} already valid"
        )

        return {
            "year": year,
            "month": month,
            "rows": len(existing),
            "file": str(output_file),
            "status": "existing_valid",
        }

    start, end = month_bounds(year, month)

    where_clause = (
        f"created_date >= '{to_soql_timestamp(start)}' "
        f"AND created_date < '{to_soql_timestamp(end)}'"
    )

    batches = []
    offset = 0
    started = time.perf_counter()

    while offset < expected_rows:
        response = session.get(
            BASE_URL,
            params={
                "$select": ",".join(COLUMNS),
                "$where": where_clause,
                "$order": (
                    "created_date ASC, "
                    "unique_key ASC"
                ),
                "$limit": BATCH_SIZE,
                "$offset": offset,
            },
            timeout=(10, 120),
        )

        response.raise_for_status()

        payload = response.json()

        if not payload:
            break

        batch = pd.DataFrame(payload)

        # Một field hoàn toàn null trong batch có thể không
        # xuất hiện trong JSON; reindex bảo đảm schema cố định.
        batch = batch.reindex(
            columns=COLUMNS
        )

        batches.append(batch)
        offset += len(batch)

        print(
            f"{year}-{month:02d} | "
            f"{offset:,}/{expected_rows:,}"
        )

        if len(batch) < BATCH_SIZE:
            break

    month_df = pd.concat(
        batches,
        ignore_index=True,
    )

    month_df = month_df.reindex(
        columns=COLUMNS
    )

    checks = validate_month(
        month_df,
        year,
        month,
        expected_rows,
    )

    month_df.to_parquet(
        output_file,
        index=False,
        compression="snappy",
    )

    elapsed = time.perf_counter() - started

    result = {
        "year": year,
        "month": month,
        "rows": len(month_df),
        "file": str(output_file),
        "size_mb": (
            output_file.stat().st_size
            / 1024**2
        ),
        "elapsed_seconds": elapsed,
        "status": "downloaded",
    }

    del month_df
    del batches
    gc.collect()

    print(
        f"DONE {year}-{month:02d}: "
        f"{result['rows']:,} rows | "
        f"{result['size_mb']:.1f} MB | "
        f"{elapsed:.1f}s"
    )

    return result

## 7. Download toàn bộ snapshot 2022–2025

`RUN_FULL_DOWNLOAD` được để `False` trong bản notebook nộp để tránh vô tình tải lại ~13.5 triệu dòng khi chỉ bấm **Run All**.

- Lần đầu tạo dữ liệu: đổi thành `True`.
- Nếu 48 file đã tồn tại: có thể giữ `False`.
- `download_month()` cũng có cơ chế skip file đã tồn tại và hợp lệ.

In [ ]:
RUN_FULL_DOWNLOAD = False

download_results = []

if RUN_FULL_DOWNLOAD:
    for row in count_table.itertuples(index=False):
        result = download_month(
            year=int(row.year),
            month=int(row.month),
            expected_rows=int(row.rows),
            overwrite=False,
        )

        download_results.append(result)

    download_summary = pd.DataFrame(
        download_results
    )

    display(download_summary)

else:
    print(
        "RUN_FULL_DOWNLOAD = False"
    )
    print(
        "No files were downloaded in this run."
    )
    print(
        "Set RUN_FULL_DOWNLOAD = True "
        "only when creating/recreating the raw snapshot."
    )

## 8. Kiểm tra inventory của 48 file raw

Cell này có thể chạy sau khi đã tải dữ liệu hoặc sau khi upload/copy các file raw vào `OUTPUT_DIR`.

In [ ]:
raw_files = sorted(
    OUTPUT_DIR.glob(
        "nyc311_*_raw.parquet"
    )
)

inventory = []

for file in raw_files:
    # Đọc riêng unique_key + created_date để validation
    # nhẹ hơn việc giữ 15 columns của nhiều file cùng lúc.
    part = pd.read_parquet(
        file,
        columns=[
            "unique_key",
            "created_date",
        ],
    )

    match = __import__("re").search(
        r"nyc311_(\d{4})_(\d{2})_raw\.parquet$",
        file.name,
    )

    if match:
        year = int(match.group(1))
        month = int(match.group(2))
    else:
        year = None
        month = None

    inventory.append(
        {
            "file": file.name,
            "year": year,
            "month": month,
            "rows": len(part),
            "unique_keys": (
                part["unique_key"]
                .astype("string")
                .nunique(dropna=False)
            ),
            "size_mb": (
                file.stat().st_size
                / 1024**2
            ),
        }
    )

    del part
    gc.collect()

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

if not inventory_df.empty:
    print(
        "\nNumber of raw partitions:",
        len(inventory_df),
    )
    print(
        "Total rows:",
        f"{inventory_df['rows'].sum():,}",
    )
    print(
        "Duplicate keys inside partitions:",
        int(
            (
                inventory_df["rows"]
                != inventory_df["unique_keys"]
            ).sum()
        ),
    )

## 9. Tải GeoJSON Community Districts

GeoJSON là dữ liệu ngoài bảng 311, dùng để ghép `community_board` với polygon phục vụ choropleth map trong dashboard.

In [ ]:
GEOJSON_URL = (
    "https://data.cityofnewyork.us/"
    "resource/5crt-au7u.geojson"
)

if Path("/kaggle/working").exists():
    GEOJSON_PATH = (
        Path("/kaggle/working")
        / "nyc_community_districts.geojson"
    )
else:
    GEOJSON_PATH = (
        Path("../data/external")
        / "nyc_community_districts.geojson"
    ).resolve()

GEOJSON_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

response = session.get(
    GEOJSON_URL,
    timeout=(10, 120),
)

print("GeoJSON HTTP status:", response.status_code)
response.raise_for_status()

GEOJSON_PATH.write_bytes(
    response.content
)

print("Saved:", GEOJSON_PATH)
print(
    "Size:",
    round(
        GEOJSON_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

## 10. Kết quả thu thập dữ liệu

Sau bước này, project sử dụng:

```text
data/raw/
├── nyc311_2022_01_raw.parquet
├── ...
├── nyc311_2025_12_raw.parquet
```

Tổng cộng **48 monthly Parquet partitions**, với snapshot tham chiếu gồm **13,506,494 service requests**.

### Vì sao dùng Parquet + Snappy?

- giữ nguyên kiểu dữ liệu tốt hơn CSV;
- đọc theo cột nhanh;
- phù hợp với Pandas/PyArrow;
- giảm dung lượng lưu trữ;
- thuận lợi cho việc xử lý từng partition.

### Quan hệ với dashboard tự động

Notebook này chịu trách nhiệm **initial collection**. Sau khi snapshot ban đầu đã có:

```text
NYC Open Data
      ↓
dashboard/refresh_data.py
      ↓
upsert các request 2022–2025 được cập nhật
      ↓
rebuild processed partitions bị ảnh hưởng
      ↓
rebuild dashboard cache
```

Nhờ vậy bài vừa giữ đúng phạm vi nghiên cứu **2022–2025**, vừa có cơ chế **đọc lại nguồn và làm mới định kỳ**.